In [0]:
storage_account = "stbankamldev"
storage_key = dbutils.secrets.get(scope="aml-scope", key="storage-access-key")

spark.conf.set(f"fs.azure.account.key.{storage_account}.blob.core.windows.net",storage_key)

raw_path = "wasbs://raw@stbankamldev.blob.core.windows.net/"

df_account_raw = spark.read.csv(f"{raw_path}accounts.csv" ,  header=True,inferSchema = True)
df_account_raw.printSchema()
df_account_raw.show(20)

In [0]:
from pyspark.sql.functions import when, col, trim, upper, lower, concat_ws

df_account_clean = df_account_raw \
    .withColumn("clean_account_status", upper(trim(col("account_status")))) \
    .withColumn("clean_account_type" , upper(trim(col("account_type")))) \
    .withColumn("clean_currency" , upper(trim(col("currency") ))) \
    .withColumn("clean_product_category",upper(trim(col("product_category")))) \
    .select("account_id", "clean_account_status", "clean_account_type","clean_currency","customer_id","clean_product_category","branch_id")
df_account_clean.printSchema()
df_account_clean.show(20)

df_account_transformed = df_account_clean \
    .withColumn("dormant_account_flag", when(col("clean_account_status") == "DORMANT",1).otherwise(0)) \
    .withColumn("wire_account_flag", when(
                                              (col("clean_account_type") == "WIRE_ACCOUNT") |
                                              (col("clean_product_category").isin("INTERNATIONAL_TRANSFER", "TRADE_FINANCE")),
                                               1
                                              ).otherwise(0)
               ) \
    .withColumn("business_account_flag", when(col("clean_product_category").isin("CORPORATE_BANKING","SMALL_BUSINESS") , 1).otherwise(0)) \
    .withColumn("offshore_account_flag", when(col("clean_product_category") == "OFFSHORE_BANKING" , 1).otherwise(0)) \
.withColumn("foreign_currency_flag" , when(col("clean_currency") != "CAD", 1).otherwise(0)) \
    .withColumn("account_risk_score", col("dormant_account_flag") + col("wire_account_flag") + col("foreign_currency_flag") +
                col("offshore_account_flag") + col("business_account_flag")) \
    .withColumn("account_risk_level", when(col("account_risk_score") >= 3 , "HIGH")
                                      .when(col("account_risk_score") >= 2 , "MEDIUM")
                                      .otherwise("LOW")                                         
                                          )   \
    .withColumn("risk_reason", concat_ws("|",
                                         when(col("dormant_account_flag") == 1 , "DORMANT"),
                                         when(col("wire_account_flag") == 1 , "HIGH_RISK_PRODUCT").
                                         when(col("business_account_flag") == 1 , "BUSINESS_ACCOUNT"),
                                         when(col("offshore_account_flag") ==1 , "OFFSHORE_ACCOUNT"),
                                         when(col("foreign_currency_flag") == 1 , "FOREIGN_CURRENCY")))
                                         
    
df_account_transformed.printSchema()
df_account_transformed.show(20)

In [0]:
silver_account_path = "wasbs://silver@stbankamldev.blob.core.windows.net/account_risk_profile"

df_account_transformed.write \
    .mode("overwrite") \
    .parquet(silver_account_path)

In [0]:
df_silver_account = spark.read.parquet(silver_account_path)
display(df_silver_account)

In [0]:
df_gold_customer_risk = spark.read.parquet("wasbs://gold@stbankamldev.blob.core.windows.net/customer_risk_score")


df_gold_customer_account_risk = df_silver_account.alias("a") \
    .join(df_gold_customer_risk.alias("c") , col("a.customer_id") == col("c.customer_id"), "inner") \
        .select( col("a.account_id"),
        col("a.customer_id"),
        col("c.customer_name"),
        col("c.clean_country_code"),
        col("c.country_risk_level"),
        col("a.clean_account_status"),
        col("a.clean_account_type"),
        col("a.clean_currency"),
        col("a.clean_product_category"),
        col("a.account_risk_score"),
        col("a.account_risk_level"),
        col("a.risk_reason").alias("account_risk_reason"),
        col("c.risk_score").alias("customer_risk_score"),
        col("c.risk_level").alias("customer_risk_level"),
        col("c.risk_reason").alias("customer_risk_reason"))


display(df_gold_customer_account_risk)
df_gold_combine = df_gold_customer_account_risk \
    .withColumn(
        "combined_risk_score",
        col("customer_risk_score") + col("account_risk_score")
    ) \
    .withColumn(
        "combined_risk_level",
        when(col("combined_risk_score") >= 5, "HIGH")
        .when(col("combined_risk_score") >= 3, "MEDIUM")
        .otherwise("LOW")
    ) 
df_gold_combine.write.mode("overwrite").parquet("wasbs://gold@stbankamldev.blob.core.windows.net/combined_risk_score")

df_gold_combine = spark.read.parquet("wasbs://gold@stbankamldev.blob.core.windows.net/combined_risk_score")
display(df_gold_combine)